In [1]:
import os, subprocess
import json
import uproot
import awkward as ak
import numpy as np
from coffea import processor, util, hist
import pandas as pd
import pickle

In [2]:
lumis = {}
lumis['2016'] = 35.9
lumis['2017'] = 41.5
lumis['2018'] = 59.9

with open('xsec.json') as f:
  xs = json.load(f)

with open('pmap.json') as f:
  pmap = json.load(f)

In [3]:
year = '2017'
nfiles = len(subprocess.getoutput("ls infiles-split/"+year+"*.json").split())
outsum = processor.dict_accumulator()

In [4]:
repickle=True

# Check if pickle exists, and don't re-create it if it does
picklename = 'pickles/'+str(year)+'_cutflow.pkl'
if os.path.isfile(picklename):
    repickle=False

In [5]:
# Load all files - this takes a while
if repickle:
    nfiles = len(subprocess.getoutput("ls infiles-split/"+year+"*.json").split())
    for n in range(1,nfiles+1):

        with open('infiles-split/'+year+'_'+str(n)+'.json') as f:
            infiles = json.load(f)
    
        filename = '/myeosdir/ggf-vbf/outfiles-ddb2/'+year+'_'+str(n)+'.coffea'
        if os.path.isfile(filename):
            out = util.load(filename)
            outsum.add(out)
        else:
            print('Missing file '+str(n),infiles.keys())
            #print("File " + filename + " is missing")
        
    scale_lumi = {k: xs[k] * 1000 *lumis[year] / w for k, w in outsum['sumw'].items()}
    outsum['cutflow'].scale(scale_lumi, 'dataset')
    cutflow = outsum['cutflow'].group('dataset', hist.Cat('process', 'Process'), pmap)

    outfile = open(picklename, 'wb')
    pickle.dump(cutflow, outfile, protocol=-1)
    outfile.close()

In [6]:
# Read the histogram from the pickle file
templates = pickle.load(open(picklename,'rb'))

In [38]:
cutflow = templates.sum('genflavor').integrate('region',['signal-ggf'])
cutflow

<Hist (process,cut) instance at 0x7fade571ff40>

In [39]:
cutflow.values()

{('ZH',): array([364.87759027, 364.87759027, 364.87759027,  95.78587868,
         95.78587868,  91.29659739,  55.44574663,  26.41843554,
         19.90507239,  16.91141534,  16.68419569,   0.44463518,
          0.        ,   0.        ,   0.        ]),
 ('WH',): array([434.78167914, 434.78167914, 434.78167914, 120.45763445,
        120.45763445, 113.8463937 ,  66.08109828,  35.40511811,
         24.623455  ,  21.50911126,  20.96230279,   0.87484646,
          0.        ,   0.        ,   0.        ]),
 ('ttH',): array([1.15673006e+03, 1.15673006e+03, 1.15673006e+03, 4.40219415e+02,
        4.40219415e+02, 3.71663145e+02, 1.30395311e+02, 4.35528574e+01,
        3.33204477e+01, 2.20658940e+01, 2.15428510e+01, 3.75066045e-01,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]),
 ('VBF',): array([752.02708561, 752.02708561, 752.02708561,  92.41193265,
         92.41193265,  89.74428462,  48.90188601,  39.8479045 ,
         37.82688288,  36.46788687,  20.9499424 ,   1.69629288,
        

In [40]:
df1 = pd.DataFrame([])

df1['ggF'] = cutflow.values()[('ggF',)]
df1['VBF'] = cutflow.values()[('VBF',)]
df1['WH'] = cutflow.values()[('WH',)]
df1['ZH'] = cutflow.values()[('ZH',)]
df1['ttH'] = cutflow.values()[('ttH',)]

df1 = df1[:-3].astype('int')
df1.index = ['nothing','trigger','lumimask','minjetkin','jetid','jetacceptance','n2ddt','antiak4btagMediumOppHem','met','noleptons','notvbf','ddbpass']
df1

,ggF,VBF,WH,ZH,ttH
nothing,1751,752,434,364,1156
trigger,1751,752,434,364,1156
lumimask,1751,752,434,364,1156
minjetkin,253,92,120,95,440
jetid,253,92,120,95,440
jetacceptance,239,89,113,91,371
n2ddt,113,48,66,55,130
antiak4btagMediumOppHem,89,39,35,26,43
met,87,37,24,19,33
noleptons,85,36,21,16,22


In [41]:
df1 = df1[3:-2]
df1.index = ['Jet kinematics','Jet ID','Jet acceptance','Jet $N_2^\text{DDT}$','Opp. hem. b veto','MET $<$ 140 GeV','No leptons']
df1.to_latex(buf=year+'/cutflow-sig.tex')
df1

,ggF,VBF,WH,ZH,ttH
Jet kinematics,253,92,120,95,440
Jet ID,253,92,120,95,440
Jet acceptance,239,89,113,91,371
Jet $N_2^\text{DDT}$,113,48,66,55,130
Opp. hem. b veto,89,39,35,26,43
MET $<$ 140 GeV,87,37,24,19,33
No leptons,85,36,21,16,22


In [42]:
df2 = pd.DataFrame([])

df2['QCD'] = cutflow.values()[('QCD',)]
df2['Wjets'] = cutflow.values()[('Wjets',)]
df2['Zjets'] = cutflow.values()[('Zjets',)]
df2['VV'] = cutflow.values()[('VV',)]
df2['ttbar'] = cutflow.values()[('ttbar',)]
df2['singlet'] = cutflow.values()[('singlet',)]

df2 = df2[:-3].astype('int')
df2.index = ['nothing','trigger','lumimask','minjetkin','jetid','jetacceptance','n2ddt','antiak4btagMediumOppHem','met','noleptons','notvbf','ddbpass']
df2

,QCD,Wjets,Zjets,VV,ttbar,singlet
nothing,360651852,2807803,984326,25062,928403,126586
trigger,360651852,2807803,984326,25062,928403,126586
lumimask,360651852,2807803,984326,25062,928403,126586
minjetkin,24028587,407040,172838,5335,232673,26222
jetid,24028587,407040,172838,5335,232673,26222
jetacceptance,22322982,368945,156225,5052,209570,24020
n2ddt,5882370,145869,69304,2888,72394,10988
antiak4btagMediumOppHem,5428393,135829,61774,2643,38493,5629
met,5311389,120380,59795,1969,29659,4372
noleptons,5167812,104015,44069,1545,20409,3375


In [43]:
df2 = df2[3:-2]
df2.index = ['Jet kinematics','Jet ID','Jet acceptance','Jet $N_2^\text{DDT}$','Opp. hem. b veto','MET $<$ 140 GeV','No leptons']
df2.to_latex(buf=year+'/cutflow-bkg.tex')

In [44]:
df2

,QCD,Wjets,Zjets,VV,ttbar,singlet
Jet kinematics,24028587,407040,172838,5335,232673,26222
Jet ID,24028587,407040,172838,5335,232673,26222
Jet acceptance,22322982,368945,156225,5052,209570,24020
Jet $N_2^\text{DDT}$,5882370,145869,69304,2888,72394,10988
Opp. hem. b veto,5428393,135829,61774,2643,38493,5629
MET $<$ 140 GeV,5311389,120380,59795,1969,29659,4372
No leptons,5167812,104015,44069,1545,20409,3375
